# 2 · Distil the teacher into a MobileNet student, export for the Raspberry Pi 3B+ (camera)

**Input:** `MyDrive/concrete_inspection/teacher/teacher_best.pth` from Notebook 1.
**Output:** `pi_package.zip`: the student model (ONNX, int8), `pi_infer.py` (photos or live camera) and `setup_pi.sh`.

**Knowledge distillation in one paragraph.** The teacher (DINOv3-L, ~300M parameters) is accurate but far too heavy for a
Pi. The student (MobileNetV3 + small decoder, ~3M parameters) is small enough. During training both see the same image and
the student is trained with three signals: (1) the **human labels**, (2) the teacher's **per-pixel class probabilities**
(e.g. "70 % crack, 25 % spalling", which carries more information than the label alone), and (3) a hint to make its
**internal features** resemble the teacher's. The teacher is frozen; only the student learns.

**Resolution trick:** the teacher sees each training crop at full resolution; the student sees the same crop at half
resolution, the size it will run at on the Pi (512 × 512 for a full image). The student learns to give
high-resolution-quality answers from a low-resolution input.

**Run order:** GPU runtime, then top to bottom. Sections: 1 setup · 2 student · 3 distillation · 4 ONNX + int8 ·
5 Pi script + camera dry-run · 6 package for the Pi.

## 1. Setup

In [1]:
import torch
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1024**3:.1f} GB, compute capability {p.major}.{p.minor}")
    print("bf16:", p.major >= 8, "(False -> DINOv3 runs in fp32: slower, more memory)")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU")

PyTorch 2.11.0+cu130 | CUDA: True
GPU: NVIDIA A100-SXM4-40GB, 39.5 GB, compute capability 8.0
bf16: True (False -> DINOv3 runs in fp32: slower, more memory)


In [4]:
!pip install -q "timm>=1.0.27" onnx onnxruntime onnxscript scikit-image opencv-python-headless

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import glob, os, shutil
DATA_ROOT = "/content/s2ds_full"

# find the uploaded dataset folder (the one that contains val/) anywhere in My Drive, up to 3 levels deep
hits = [os.path.dirname(p) for p in
        glob.glob("/content/drive/MyDrive/*/val") + glob.glob("/content/drive/MyDrive/*/*/val")
        + glob.glob("/content/drive/MyDrive/*/*/*/val")
        if os.path.exists(os.path.join(os.path.dirname(p), "train"))]
print("found:", hits)

if not hits:
    raise FileNotFoundError("Dataset folder not found. Please ensure it is uploaded to Google Drive and that Drive is mounted.")

src = hits[0]

shutil.rmtree(DATA_ROOT, ignore_errors=True)
!rm -f /content/s2ds.zip
print("copying from Drive (a few minutes)...")
shutil.copytree(src, DATA_ROOT)
for split in ("train", "val", "test"):
    print(split, len(glob.glob(f"{DATA_ROOT}/{split}/*_lab.png")), "labelled images")


found: ['/content/drive/MyDrive/concrete_inspection/s2ds']
copying from Drive (a few minutes)...
train 563 labelled images
val 87 labelled images
test 93 labelled images


In [7]:
import os, sys

DRIVE_DIR = "/content/drive/MyDrive/concrete_inspection"      # everything we keep goes here
REPO      = "/content/s2ds-defect-segmentation"
DATA_ROOT = "/content/s2ds_full"

# 1. code library: model.py (DINOv3-L teacher), s2ds_data.py (dataset + official metric), losses.py
if not os.path.exists(REPO):
    !git clone -q https://github.com/A-SHOJAEI/s2ds-defect-segmentation.git {REPO}

# 2. Google Drive (checkpoints survive Colab disconnects there)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

# 3. S2DS dataset. A copy of the zip is kept in YOUR Drive, so the public link (which has a
#    download quota: "Too many users have viewed or downloaded this file") is needed only once.
DATA_ZIP = f"{DRIVE_DIR}/s2ds_dataset.zip"
if not os.path.exists(DATA_ROOT + "/val"):
    if os.path.exists(DATA_ZIP):
        !cp "{DATA_ZIP}" /content/s2ds_dataset.zip
    else:
        !gdown 1PQ50QKfy2vnDOHSmw5bpBFi33hZsSXuM -O /content/s2ds_dataset.zip
        if os.path.exists("/content/s2ds_dataset.zip"):
            !cp /content/s2ds_dataset.zip "{DATA_ZIP}"
    if not os.path.exists("/content/s2ds_dataset.zip"):
        raise SystemExit("Dataset download blocked. Open https://drive.google.com/uc?id=1PQ50QKfy2vnDOHSmw5bpBFi33hZsSXuM "
                         f"in your browser, download the zip, upload it to Drive as {DATA_ZIP}, then re-run this cell.")
    !unzip -q /content/s2ds_dataset.zip -d {DATA_ROOT}

# 4. teacher checkpoint from Notebook 1 (copied locally: faster to read)
TEACHER_ON_DRIVE = f"{DRIVE_DIR}/teacher/teacher_best.pth"
TEACHER_CKPT = "/content/teacher_best.pth"
if not os.path.exists(TEACHER_CKPT):
    !cp "{TEACHER_ON_DRIVE}" {TEACHER_CKPT}
mb = os.path.getsize(TEACHER_CKPT) / 1e6
print(f"teacher checkpoint: {mb:.0f} MB ->", "head only (backbone = pretrained DINOv3-L)" if mb < 100
      else "full fine-tuned model")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
teacher checkpoint: 1230 MB -> full fine-tuned model


### Configuration
`SCALE=0.5` → the Pi runs the model at 512 × 512. `CROP` is the student's training crop; the teacher sees
`CROP / SCALE` = 768 px (must be a multiple of 16).

In [8]:
SCALE   = 0.5            # deploy size = 1024 * SCALE = 512 px
CROP    = 384            # student training crop (teacher sees CROP / SCALE = 768 px)
EPOCHS  = 25
STUDENT_DIR = f"{DRIVE_DIR}/student"
PI_DIR  = "/content/pi_package"
assert round(CROP / SCALE) % 16 == 0


## 2. Student model

In [9]:
%%writefile student_model.py
"""STUDENT segmentation model: MobileNetV3-Large encoder + light FPN decoder (~3M params).

Small and fast enough for a Raspberry Pi 3B+, and it predicts at output stride 4
(a 512px input gives a 128x128 label map), which keeps thin cracks visible.

forward(x, upsample=True, return_feat=False)
  upsample=True  -> logits at the input resolution (training / evaluation)
  upsample=False -> low-resolution logits (what we export to ONNX; the Pi upsamples cheaply)
  return_feat    -> also return the stride-16 decoder feature projected to the teacher's
                    1024-d feature space (used ONLY for the feature-distillation loss;
                    `feat_proj` is not part of the exported model).

Run `python student_model.py` for a quick shape/parameter self-test (no downloads needed).
"""
from __future__ import annotations

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES = 7
TEACHER_FEAT_DIM = 1024          # DINOv3-L embedding width


def _cba(cin, cout, k=1, groups=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, 1, k // 2, groups=groups, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


def _dwsep(cin, cout):
    """Depthwise-separable 3x3 conv (cheap, quantization-friendly)."""
    return nn.Sequential(_cba(cin, cin, 3, groups=cin), _cba(cin, cout, 1))


class StudentSeg(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, hid: int = 96, pretrained: bool = True,
                 drop: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        # ImageNet-pretrained encoder; features at strides 4, 8, 16, 32
        self.enc = timm.create_model("mobilenetv3_large_100.ra_in1k", pretrained=pretrained,
                                     features_only=True, out_indices=(1, 2, 3, 4))
        chs = self.enc.feature_info.channels()                           # [24, 40, 112, 960]
        self.lat = nn.ModuleList([_cba(c, hid, 1) for c in chs])
        self.smooth = nn.ModuleList([_dwsep(hid, hid) for _ in range(len(chs) - 1)])
        self.head = nn.Sequential(_dwsep(hid, hid), nn.Dropout2d(drop), nn.Conv2d(hid, num_classes, 1))
        # training-only: map the stride-16 decoder feature to the teacher's feature space
        self.feat_proj = nn.Conv2d(hid, TEACHER_FEAT_DIM, 1)

    def forward(self, x, upsample: bool = True, return_feat: bool = False):
        f = self.enc(x)                                       # strides 4, 8, 16, 32
        lat = [m(fi) for m, fi in zip(self.lat, f)]
        p, feat16 = lat[3], None
        for i in (2, 1, 0):                                   # top-down: s16, s8, s4
            p = F.interpolate(p, size=lat[i].shape[-2:], mode="bilinear", align_corners=False) + lat[i]
            p = self.smooth[2 - i](p)
            if i == 2:
                feat16 = p
        logits = self.head(p)                                 # stride 4
        if upsample:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        if return_feat:
            return logits, self.feat_proj(feat16)
        return logits

    def param_groups(self, enc_lr: float, head_lr: float):
        enc = list(self.enc.parameters())
        enc_ids = {id(p) for p in enc}
        rest = [p for p in self.parameters() if id(p) not in enc_ids]
        return [{"params": enc, "lr": enc_lr}, {"params": rest, "lr": head_lr}]

    def deployable_params(self) -> int:
        return sum(p.numel() for n, p in self.named_parameters() if not n.startswith("feat_proj."))


if __name__ == "__main__":
    m = StudentSeg(pretrained=False).eval()
    x = torch.randn(1, 3, 512, 512)
    with torch.no_grad():
        full, low = m(x), m(x, upsample=False)
        _, ft = m(x, return_feat=True)
    print(f"deployable params={m.deployable_params() / 1e6:.1f}M | full {tuple(full.shape)} "
          f"low-res {tuple(low.shape)} feature {tuple(ft.shape)}")
    assert full.shape == (1, NUM_CLASSES, 512, 512) and low.shape == (1, NUM_CLASSES, 128, 128)
    assert ft.shape[1] == TEACHER_FEAT_DIM
    print("OK")

Writing student_model.py


In [10]:
!python student_model.py

deployable params=3.1M | full (1, 7, 512, 512) low-res (1, 7, 128, 128) feature (1, 1024, 32, 32)
OK


## 3. Distillation
First a **smoke test** (5 steps, 5 val images, a few minutes) to catch problems early. Read the `checkpoint:` line: it
says whether the teacher checkpoint holds the whole model or only the head.

The full run first prints the **teacher's validation mIoU** (the target), trains the student, then prints a per-class
**teacher vs student** table. Expect the student to be somewhat below the teacher; Crack usually loses the most.
The teacher runs at 768 px on every step, so a bf16 GPU (A100/L4) is much faster than a T4.

In [11]:
%%writefile distill.py
"""Distil the fine-tuned DINOv3-L TEACHER into the small MobileNetV3 STUDENT.

The teacher is DINOv3MultiScaleSeg(num_classes=7) (repo's model.py) + the checkpoint produced by
the teacher-training notebook (train_teacher.py). The script reports what the checkpoint contains.

Training signal for the student (all computed on the SAME augmented crop):
    1. hard loss     focal+dice against the human labels           (repo's losses.py)
    2. logit KD      KL(teacher || student) on softened logits     (teaches class relationships)
    3. feature KD    student stride-16 feature -> teacher's last-layer feature (MSE on LayerNormed maps)

Resolution trick (--scale): the teacher sees the crop at NATIVE resolution (so it can still see
hairline cracks), the student sees the same crop DOWNSCALED by `scale` (the resolution it will run
at on the Pi). The student therefore learns to reproduce high-res teacher quality from a
low-res input.   crop=384, scale=0.5  ->  teacher input 768px, student input 384px, Pi input 512px.

Evaluation is done at full 1024px against the official-protocol labels (repo's s2ds_data.py), so
student and teacher numbers are directly comparable.

Example (Colab):
    python distill.py --repo /content/s2ds-defect-segmentation --data_root /content/s2ds_full \
        --teacher_ckpt /content/teacher_best.pth
"""
from __future__ import annotations

import argparse
import copy
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

from student_model import NUM_CLASSES, TEACHER_FEAT_DIM, StudentSeg


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================================ repo + teacher
def import_repo(repo: str, data_root: str | None):
    """Import the repo's modules. S2DS_ROOT must be set BEFORE importing s2ds_data."""
    if data_root:
        os.environ["S2DS_ROOT"] = data_root
    sys.path.insert(0, repo)
    import s2ds_data as D
    from losses import FocalDiceLoss, class_weights_from_counts
    from model import DINOv3MultiScaleSeg
    return D, FocalDiceLoss, class_weights_from_counts, DINOv3MultiScaleSeg


def read_checkpoint(path: str) -> dict:
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, torch.nn.Module):               # torch.save(model) instead of a state-dict
        ck = ck.state_dict()
    sd = ck
    if isinstance(ck, dict):
        for k in ("ema", "model", "state_dict"):        # same keys the notebook / train.py use
            if k in ck and isinstance(ck[k], dict):
                sd = ck[k]
                break
    return {k.removeprefix("module."): v for k, v in sd.items()}


def build_teacher(DINOv3MultiScaleSeg, ckpt_path: str, image_size: int, device):
    sd = read_checkpoint(ckpt_path)
    has_bb = any(k.startswith("backbone.") for k in sd)
    has_head = any(k.startswith("head.") for k in sd)
    n_bb = sum(k.startswith("backbone.") for k in sd)
    n_hd = sum(k.startswith("head.") for k in sd)
    log(f"checkpoint: {len(sd)} tensors | backbone.*={n_bb} head.*={n_hd}")

    # Only download the (gated) generic DINOv3 weights if the checkpoint doesn't carry a backbone.
    teacher = DINOv3MultiScaleSeg(num_classes=NUM_CLASSES, image_size=image_size,
                                  pretrained=not has_bb)
    if not has_bb and not has_head:
        # looks like a bare head state-dict ("0.weight", "1.weight", ...)
        teacher.head.load_state_dict(sd, strict=True)
        log("checkpoint is a bare head state-dict -> loaded into teacher.head (strict)")
        log("NOTE: teacher backbone = generic pretrained DINOv3-L (NOT fine-tuned).")
    else:
        missing, unexpected = teacher.load_state_dict(sd, strict=False)
        head_missing = [k for k in missing if k.startswith("head.")]
        if head_missing:
            raise RuntimeError(f"checkpoint is missing head weights {head_missing[:4]}... -> the "
                               "teacher head would be RANDOM. Wrong checkpoint / wrong architecture?")
        if unexpected:
            log(f"WARNING: {len(unexpected)} unexpected keys in checkpoint, e.g. {unexpected[:3]}")
        if not has_bb:
            log("NOTE: checkpoint has no backbone weights -> teacher backbone = pretrained DINOv3-L "
                "(expected if the teacher was trained with --train_blocks 0, i.e. head only).")
    teacher = teacher.to(device).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    return teacher


@torch.no_grad()
def teacher_outputs(teacher, x, out_hw, amp_dtype):
    """One teacher pass -> (logits resized to out_hw, last-layer feature map).
    Mirrors DINOv3MultiScaleSeg.forward but also returns the features."""
    assert x.shape[-1] % 16 == 0 and x.shape[-2] % 16 == 0, "teacher input must be a multiple of 16"
    if amp_dtype is not None:
        with torch.autocast("cuda", dtype=amp_dtype):
            feat = teacher._encode(x)
    else:
        feat = teacher._encode(x)
    feat = feat.float()
    logits = teacher.head(feat)
    logits = F.interpolate(logits, size=tuple(out_hw), mode="bilinear", align_corners=False)
    return logits, feat[:, -TEACHER_FEAT_DIM:]


# ============================================================================ data
class ScaledS2DS(Dataset):
    """Wraps the repo's S2DSDataset.

    train -> (x_native_crop, x_scaled_crop, y_scaled_crop)
    eval  -> (x_native_full, x_scaled_full, y_native_full)   (labels keep the official ignore band)
    """

    def __init__(self, D, split: str, train: bool, crop: int, scale: float):
        self.scale, self.train = scale, train
        native_crop = int(round(crop / scale))
        assert native_crop % 16 == 0, f"crop/scale={native_crop} must be a multiple of 16"
        self.base = D.S2DSDataset(split, crop=native_crop, train=train)

    def __len__(self):
        return len(self.base)

    def _down(self, x):
        hs, ws = int(round(x.shape[-2] * self.scale)), int(round(x.shape[-1] * self.scale))
        return F.interpolate(x[None], size=(hs, ws), mode="bilinear", align_corners=False,
                             antialias=True)[0]

    def __getitem__(self, i):
        x, y = self.base[i]
        xs = self._down(x) if self.scale != 1.0 else x
        if self.train:
            ys = y
            if self.scale != 1.0:
                ys = F.interpolate(y[None, None].float(), size=xs.shape[-2:], mode="nearest")[0, 0].long()
            return x, xs, ys
        return x, xs, y


def _worker_init(_):
    import cv2
    cv2.setNumThreads(0)
    torch.set_num_threads(1)


def class_counts(D, out_dir: Path):
    path = out_dir / "train_class_counts.json"
    if path.exists():
        return json.load(open(path))
    log("counting train-set class pixels (one-off, ~1 min)...")
    c = np.zeros(NUM_CLASSES, np.float64)
    for _, lab in D.list_split("train"):
        c += np.bincount(D.lab2class(D._read_rgb(lab)).ravel(), minlength=NUM_CLASSES)[:NUM_CLASSES]
    json.dump(c.tolist(), open(path, "w"))
    return c.tolist()


# ============================================================================ losses
def kd_loss(s_logits, t_logits, T: float):
    s = F.log_softmax(s_logits / T, dim=1)
    t = F.softmax(t_logits / T, dim=1)
    return F.kl_div(s, t, reduction="none").sum(1).mean() * (T * T)


def _ln(x):
    return F.layer_norm(x.permute(0, 2, 3, 1), (x.shape[1],)).permute(0, 3, 1, 2)


def feat_loss(s_feat, t_feat):
    t_feat = F.adaptive_avg_pool2d(t_feat, s_feat.shape[-2:])
    return F.mse_loss(_ln(s_feat), _ln(t_feat))


# ============================================================================ evaluation
@torch.no_grad()
def evaluate(D, predict_fn, ds, device, desc=""):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), np.float64)
    for x_nat, x_s, y_nat in DataLoader(ds, batch_size=1, shuffle=False, num_workers=2,
                                        worker_init_fn=_worker_init):
        logits = predict_fn(x_nat.to(device), x_s.to(device), y_nat.shape[-2:])
        pred = logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
        cm += D.confusion_from(pred.ravel(), y_nat[0].numpy().astype(np.uint8).ravel())
    iou = D.iou_from_confusion(cm)
    res = {"per_class_iou": {n: (None if np.isnan(v) else float(v)) for n, v in zip(D.CLASS_NAMES, iou)},
           "miou_all7": float(np.nanmean(iou)),
           "miou_defect_only": float(np.nanmean(iou[D.DEFECT_IDS]))}
    log(f"  [{desc}] mIoU(all7)={res['miou_all7']:.4f} defect-only={res['miou_defect_only']:.4f} | "
        + " ".join(f"{k[:5]}:{(v or 0):.2f}" for k, v in res["per_class_iou"].items()))
    return res


# ============================================================================ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", default="/content/s2ds-defect-segmentation")
    ap.add_argument("--data_root", default="/content/s2ds_full")
    ap.add_argument("--teacher_ckpt", default="/content/teacher_best.pth")
    ap.add_argument("--hid", type=int, default=96, help="decoder width")
    ap.add_argument("--crop", type=int, default=384, help="STUDENT training crop (px)")
    ap.add_argument("--scale", type=float, default=0.5,
                    help="student resolution / native resolution (deploy size = 1024*scale)")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--enc_lr", type=float, default=3e-4)
    ap.add_argument("--head_lr", type=float, default=1e-3)
    ap.add_argument("--kd_T", type=float, default=2.0)
    ap.add_argument("--w_hard", type=float, default=1.0)
    ap.add_argument("--w_kd", type=float, default=1.0)
    ap.add_argument("--w_feat", type=float, default=0.25)
    ap.add_argument("--ema_decay", type=float, default=0.999)
    ap.add_argument("--eval_every", type=int, default=5)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="./student_runs")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--skip_teacher_eval", action="store_true")
    ap.add_argument("--max_steps", type=int, default=0, help="smoke test: stop after N steps")
    ap.add_argument("--max_eval_images", type=int, default=0, help="smoke test: limit val images")
    args = ap.parse_args()

    tag = args.tag or "student"
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cc = torch.cuda.get_device_capability()[0] if device.type == "cuda" else 0
    amp = torch.bfloat16 if cc >= 8 else None          # fp16 NaNs on DINOv3; no bf16 on T4 -> fp32
    log(f"device={device} amp={'bf16' if amp else 'fp32 (no bf16 on this GPU; slower)'}")

    D, FocalDiceLoss, class_weights_from_counts, DINOv3MultiScaleSeg = import_repo(args.repo, args.data_root)
    native_crop = int(round(args.crop / args.scale))

    # ---- teacher
    teacher = build_teacher(DINOv3MultiScaleSeg, args.teacher_ckpt, native_crop, device)

    # ---- data
    train_ds = ScaledS2DS(D, "train", True, args.crop, args.scale)
    val_ds = ScaledS2DS(D, "val", False, args.crop, args.scale)
    if args.max_eval_images:
        val_ds = Subset(val_ds, list(range(min(args.max_eval_images, len(val_ds)))))
    loader = DataLoader(train_ds, batch_size=args.batch, shuffle=True, num_workers=args.workers,
                        drop_last=True, pin_memory=device.type == "cuda", worker_init_fn=_worker_init)
    log(f"train images={len(train_ds)} steps/epoch={len(loader)} | teacher sees {native_crop}px, "
        f"student sees {args.crop}px, deploy size {int(round(1024 * args.scale))}px")

    # ---- teacher baseline (the number the student is trying to match)
    results = {"args": vars(args)}
    if not args.skip_teacher_eval:
        log("evaluating TEACHER on val (full 1024px)...")
        results["teacher_val"] = evaluate(
            D, lambda xn, xs, hw: teacher_outputs(teacher, xn, hw, amp)[0], val_ds, device, "teacher")

    # ---- student
    student = StudentSeg(hid=args.hid, pretrained=True).to(device)
    ema = copy.deepcopy(student).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    log(f"student=MobileNetV3 deployable params={student.deployable_params() / 1e6:.1f}M")

    opt = torch.optim.AdamW(student.param_groups(args.enc_lr, args.head_lr), weight_decay=0.01)
    total = args.epochs * len(loader)
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))

    cw = class_weights_from_counts(class_counts(D, out), power=0.5).to(device)
    crit = FocalDiceLoss(num_classes=NUM_CLASSES, class_weights=cw, ignore_index=D.IGNORE_ID).to(device)

    def student_predict(model):
        def fn(x_nat, x_s, hw):
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                lg = model(x_s)
            return F.interpolate(lg.float(), size=tuple(hw), mode="bilinear", align_corners=False)
        return fn

    best, step = {"miou_all7": -1.0}, 0
    for ep in range(args.epochs):
        student.train()
        t0, agg, n_steps = time.time(), np.zeros(4), 0
        for x_nat, x_s, y_s in loader:
            x_nat, x_s, y_s = x_nat.to(device, non_blocking=True), x_s.to(device, non_blocking=True), \
                y_s.to(device, non_blocking=True)
            t_logits, t_feat = teacher_outputs(teacher, x_nat, x_s.shape[-2:], amp)
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                s_logits, s_feat = student(x_s, return_feat=True)
            s_logits, s_feat = s_logits.float(), s_feat.float()

            l_hard = crit(s_logits, y_s)
            l_kd = kd_loss(s_logits, t_logits, args.kd_T)
            l_ft = feat_loss(s_feat, t_feat) if args.w_feat > 0 else s_logits.sum() * 0
            loss = args.w_hard * l_hard + args.w_kd * l_kd + args.w_feat * l_ft

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step(); sched.step(); step += 1

            d = min(args.ema_decay, (1 + step) / (10 + step))
            with torch.no_grad():
                for e, m in zip(ema.parameters(), student.parameters()):
                    e.mul_(d).add_(m.detach(), alpha=1 - d)
                for e, m in zip(ema.buffers(), student.buffers()):
                    e.copy_(m)
            agg += np.array([loss.item(), l_hard.item(), l_kd.item(), l_ft.item()])
            n_steps += 1
            if args.max_steps and step >= args.max_steps:
                break
        agg /= max(1, n_steps)
        log(f"epoch {ep + 1}/{args.epochs} loss={agg[0]:.3f} (hard {agg[1]:.3f} kd {agg[2]:.3f} "
            f"feat {agg[3]:.3f}) {time.time() - t0:.0f}s")

        done = (ep + 1 == args.epochs) or (args.max_steps and step >= args.max_steps)
        if (ep + 1) % args.eval_every == 0 or done:
            r = evaluate(D, student_predict(ema), val_ds, device, "student-ema")
            if r["miou_all7"] > best["miou_all7"]:
                best = r
                torch.save({"ema": ema.state_dict(),
                            "cfg": {"hid": args.hid, "crop": args.crop,
                                    "scale": args.scale, "num_classes": NUM_CLASSES},
                            "val": r}, out / f"{tag}_best.pt")
                log(f"  -> saved {out / (tag + '_best.pt')}")
        if done:
            break

    results["student_val_best"] = best
    json.dump(results, open(out / f"{tag}_result.json", "w"), indent=2)
    if "teacher_val" in results:
        t, s = results["teacher_val"], best
        log("=== teacher vs student (val, official-protocol IoU) ===")
        for k in t["per_class_iou"]:
            log(f"  {k:14s} teacher {t['per_class_iou'][k] or 0:.3f}   student {s['per_class_iou'][k] or 0:.3f}")
        log(f"  {'mIoU(all7)':14s} teacher {t['miou_all7']:.3f}   student {s['miou_all7']:.3f}")


if __name__ == "__main__":
    main()

Writing distill.py


In [12]:
!python distill.py --repo {REPO} --data_root {DATA_ROOT} --teacher_ckpt {TEACHER_CKPT} \
    --crop {CROP} --scale {SCALE} --tag smoke --max_steps 5 --max_eval_images 5 --skip_teacher_eval \
    --out /content/student_smoke

[14:49:57] device=cuda amp=bf16
[14:49:58] checkpoint: 332 tensors | backbone.*=318 head.*=14
[14:50:03] train images=563 steps/epoch=70 | teacher sees 768px, student sees 384px, deploy size 512px

model.safetensors: downloading bytes:  72% 16.0M/22.1M [00:01<00:00, 17.7MB/s,  606kB/s  ]
model.safetensors: downloading bytes: 100% 20.8M/20.8M [00:01<00:00, 14.1MB/s, 1.99MB/s  ]
model.safetensors: reconstructing file: 100% 22.1M/22.1M [00:01<00:00, 14.9MB/s, 2.12MB/s  ]
Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
[14:50:05] student=MobileNetV3 deployable params=3.1M
[14:50:05] counting train-set class pixels (one-off, ~1 min)...
[14:52:08] epoch 1/40 loss=9.723 (hard 1.485 kd 7.736 feat 2.012) 9s
[14:52:09]   [student-ema] mIoU(all7)=0.0112 defect-only=0.0134 | backg:0.00 crack:0.00 spall:0.00 corro:0.00 efflo:0.00 veget:0.07 contr:0.00
[14:52:09]   -> saved 

In [13]:
!python distill.py --repo {REPO} --data_root {DATA_ROOT} --teacher_ckpt {TEACHER_CKPT} \
    --crop {CROP} --scale {SCALE} --epochs {EPOCHS} --tag student --out {STUDENT_DIR}

[14:52:21] device=cuda amp=bf16
[14:52:22] checkpoint: 332 tensors | backbone.*=318 head.*=14
[14:52:27] train images=563 steps/epoch=70 | teacher sees 768px, student sees 384px, deploy size 512px
[14:52:27] evaluating TEACHER on val (full 1024px)...
[14:52:39]   [teacher] mIoU(all7)=0.7855 defect-only=0.7528 | backg:0.98 crack:0.20 spall:0.88 corro:0.87 efflo:0.84 veget:0.74 contr:0.99
Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
[14:52:40] student=MobileNetV3 deployable params=3.1M
[14:53:19] epoch 1/25 loss=8.253 (hard 1.284 kd 6.588 feat 1.523) 38s
[14:53:57] epoch 2/25 loss=3.548 (hard 0.925 kd 2.413 feat 0.841) 38s
[14:54:35] epoch 3/25 loss=1.787 (hard 0.689 kd 0.919 feat 0.718) 38s
[14:55:14] epoch 4/25 loss=1.558 (hard 0.653 kd 0.741 feat 0.656) 38s
[14:55:52] epoch 5/25 loss=1.521 (hard 0.641 kd 0.731 feat 0.596) 39s
[14:56:06]   [student-ema] mIoU

In [14]:
!python distill.py --repo {REPO} --data_root {DATA_ROOT} --teacher_ckpt {TEACHER_CKPT} \
    --crop {CROP} --scale {SCALE} --epochs {EPOCHS} --tag student --out {STUDENT_DIR}

[15:10:00] device=cuda amp=bf16
[15:10:01] checkpoint: 332 tensors | backbone.*=318 head.*=14
[15:10:06] train images=563 steps/epoch=70 | teacher sees 768px, student sees 384px, deploy size 512px
[15:10:06] evaluating TEACHER on val (full 1024px)...
[15:10:18]   [teacher] mIoU(all7)=0.7855 defect-only=0.7528 | backg:0.98 crack:0.20 spall:0.88 corro:0.87 efflo:0.84 veget:0.74 contr:0.99
Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
[15:10:19] student=MobileNetV3 deployable params=3.1M
[15:10:57] epoch 1/25 loss=8.254 (hard 1.284 kd 6.589 feat 1.523) 38s
[15:11:36] epoch 2/25 loss=3.550 (hard 0.926 kd 2.414 feat 0.842) 38s
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloade

In [ ]:
!python distill.py --repo {REPO} --data_root {DATA_ROOT} --teacher_ckpt {TEACHER_CKPT} \
    --crop {CROP} --scale {SCALE} --epochs {EPOCHS} --tag student --out {STUDENT_DIR}

## 4. Export to ONNX + int8
ONNX is a portable model file that `onnxruntime` runs on the Pi without PyTorch. int8 quantization stores weights and
activations as 8-bit integers: ~4× smaller and usually faster on ARM CPUs, at a small accuracy cost (calibrated on 100
training images). Both versions are evaluated on the full validation set: compare their mIoU lines.

In [ ]:
%%writefile export_onnx.py
"""Export the distilled student to ONNX, quantize to int8, and measure accuracy + size.

Outputs (in --out_dir):
    student_fp32.onnx          float32 graph
    student_int8.onnx          int8 graph  (static QDQ quantization, calibrated on train images)
    student_meta.json          input size / normalisation / class names (read by pi_infer.py)

The exported graph takes a fixed (1,3,S,S) image and returns LOW-resolution logits
(stride 4); pi_infer.py upsamples + argmaxes with OpenCV.

Example:
    python export_onnx.py --ckpt student_runs/student_best.pt \
        --repo /content/s2ds-defect-segmentation --data_root /content/s2ds_full
"""
from __future__ import annotations

import argparse
import inspect
import json
import os
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from student_model import StudentSeg

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASS_NAMES = ["Background", "Crack", "Spalling", "Corrosion", "Efflorescence", "Vegetation", "Control Point"]


class ExportWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(x, upsample=False)


def load_student(ckpt_path):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ck["cfg"]
    m = StudentSeg(num_classes=cfg["num_classes"], hid=cfg["hid"], pretrained=False)
    m.load_state_dict(ck["ema"])
    return m.eval(), cfg


def export_fp32(model, size, path):
    dummy = torch.randn(1, 3, size, size)
    kw = {}
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        kw["dynamo"] = False                     # classic exporter: most predictable for timm models
    torch.onnx.export(ExportWrapper(model), dummy, str(path), opset_version=17,
                      input_names=["image"], output_names=["logits_lowres"], **kw)
    try:
        import onnx
        from onnxsim import simplify
        m, ok = simplify(onnx.load(str(path)))
        if ok:
            onnx.save(m, str(path))
            print("  onnx-simplifier applied")
    except Exception as e:                       # optional step
        print(f"  (skipping onnx-simplifier: {type(e).__name__})")


def calib_arrays(D, size, n):
    ds = D.S2DSDataset("train", train=False)
    idx = np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
    out = []
    for i in idx:
        x, _ = ds[int(i)]
        x = F.interpolate(x[None], size=(size, size), mode="bilinear", align_corners=False, antialias=True)
        out.append(x.numpy().astype(np.float32))
    return out


def quantize(fp32_path, int8_path, mode, D, size, calib_n):
    from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                          quantize_dynamic, quantize_static)
    if mode == "dynamic":
        quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QInt8)
        return

    class Reader(CalibrationDataReader):
        def __init__(self, arrs):
            self.it = iter([{"image": a} for a in arrs])

        def get_next(self):
            return next(self.it, None)

    src = str(fp32_path)
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        pre = str(fp32_path).replace(".onnx", "_pre.onnx")
        quant_pre_process(src, pre)
        src = pre
    except Exception as e:
        print(f"  (skipping quant pre-process: {type(e).__name__})")
    print(f"  calibrating on {calib_n} train images...")
    quantize_static(src, str(int8_path), Reader(calib_arrays(D, size, calib_n)),
                    quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)
    if src != str(fp32_path):
        os.remove(src)                           # intermediate file, not needed on the Pi


def eval_onnx(path, D, size, n, threads=4):
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    ds = D.S2DSDataset("val", train=False)
    idx = range(len(ds)) if n <= 0 else np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
    cm = np.zeros((7, 7))
    lat = []
    for i in idx:
        x, y = ds[int(i)]
        xs = F.interpolate(x[None], size=(size, size), mode="bilinear", align_corners=False,
                           antialias=True).numpy().astype(np.float32)
        t0 = time.perf_counter()
        low = sess.run(None, {"image": xs})[0][0]
        lat.append(time.perf_counter() - t0)
        H, W = y.shape
        up = np.stack([cv2.resize(c, (W, H), interpolation=cv2.INTER_LINEAR) for c in low])
        cm += D.confusion_from(up.argmax(0).astype(np.uint8).ravel(), y.numpy().astype(np.uint8).ravel())
    iou = D.iou_from_confusion(cm)
    return {"miou_all7": float(np.nanmean(iou)),
            "per_class": {n_: (None if np.isnan(v) else round(float(v), 4)) for n_, v in zip(D.CLASS_NAMES, iou)},
            "ms_per_image_this_machine": 1000 * float(np.mean(lat[1:] or lat))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--repo", default="/content/s2ds-defect-segmentation")
    ap.add_argument("--data_root", default="/content/s2ds_full")
    ap.add_argument("--out_dir", default="./pi_package")
    ap.add_argument("--size", type=int, default=0, help="deploy input size; default = 1024*scale")
    ap.add_argument("--quant", choices=["static", "dynamic", "none"], default="static")
    ap.add_argument("--calib_n", type=int, default=100)
    ap.add_argument("--eval_n", type=int, default=0, help="val images to evaluate (0 = all)")
    args = ap.parse_args()

    os.environ["S2DS_ROOT"] = args.data_root
    sys.path.insert(0, args.repo)
    import s2ds_data as D

    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    ck_cfg = torch.load(args.ckpt, map_location="cpu", weights_only=False)["cfg"]
    size = args.size or int(round(1024 * ck_cfg["scale"] / 32) * 32)
    model, cfg = load_student(args.ckpt)
    mode = args.quant
    print(f"student=MobileNetV3 deploy size={size} quant={mode}")

    fp32, int8 = out / "student_fp32.onnx", out / "student_int8.onnx"
    print("exporting fp32 ONNX..."); export_fp32(model, size, fp32)
    if mode != "none":
        print("quantizing..."); quantize(fp32, int8, mode, D, size, args.calib_n)

    json.dump({"size": size, "mean": MEAN, "std": STD, "class_names": CLASS_NAMES,
               "backbone": "mobilenetv3_large", "output": "low-res logits (1,7,h,w); upsample to size x size"},
              open(out / "student_meta.json", "w"), indent=2)

    print("\n=== results (val set, official-protocol IoU, evaluated at full 1024px) ===")
    for name, p in (("fp32", fp32), ("int8", int8)):
        if p.exists():
            r = eval_onnx(p, D, size, args.eval_n)
            print(f"{name}: {p.stat().st_size / 1e6:6.1f} MB | mIoU(all7)={r['miou_all7']:.4f} | "
                  f"{r['ms_per_image_this_machine']:.0f} ms/img on THIS machine (not the Pi)")
            print("      " + " ".join(f"{k[:5]}:{(v or 0):.2f}" for k, v in r["per_class"].items()))
    print(f"\nCopy the '{out}' folder + pi_infer.py to the Raspberry Pi.")


if __name__ == "__main__":
    main()

In [ ]:
!python export_onnx.py --ckpt {STUDENT_DIR}/student_best.pt --repo {REPO} --data_root {DATA_ROOT} --out_dir {PI_DIR}

## 5. Raspberry Pi script
`pi_infer.py` needs only `onnxruntime`, `numpy`, `opencv` (+ `picamera2` for the Pi camera module). It handles photos,
the Pi camera, a USB webcam, or a video file, saves overlays + a `log.csv` of defect areas, and can serve a live view to a
browser (`--serve 8080`), because Raspberry Pi OS Lite has no screen.

In [ ]:
%%writefile pi_infer.py
"""Concrete-defect inspection on the Raspberry Pi with the distilled student (ONNX).
Needs only onnxruntime, numpy, opencv (+ picamera2 for the Pi camera module).

Photos:
    python3 pi_infer.py photo.jpg folder_of_photos/
Camera (Pi camera module if present, otherwise the first USB webcam):
    python3 pi_infer.py --camera
    python3 pi_infer.py --camera --serve 8080      # live view at http://<pi-address>:8080
    python3 pi_infer.py --camera usb:0             # force a USB webcam
    python3 pi_infer.py --camera clip.mp4          # a video file (handy for testing)

Results go to --out: <name>_overlay.jpg (+ the raw frame for camera captures) and log.csv with
the per-defect area percentages. Frames of any aspect ratio are letterboxed to the model's
square input (no distortion).
"""
from __future__ import annotations

import argparse
import csv
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

PALETTE_RGB = np.array([[35, 35, 35], [255, 50, 50], [255, 165, 0], [160, 32, 240],
                        [255, 255, 0], [0, 200, 0], [0, 191, 255]], np.uint8)
PALETTE_BGR = PALETTE_RGB[:, ::-1].copy()


# ============================================================================ model
class Segmenter:
    def __init__(self, model_path, meta_path=None, threads=4):
        meta = json.load(open(meta_path or Path(model_path).with_name("student_meta.json")))
        self.size, self.names = meta["size"], meta["class_names"]
        self.mean = np.array(meta["mean"], np.float32)
        self.std = np.array(meta["std"], np.float32)
        so = ort.SessionOptions()
        so.intra_op_num_threads = threads
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.sess = ort.InferenceSession(str(model_path), so, providers=["CPUExecutionProvider"])

    def letterbox(self, bgr):
        h, w = bgr.shape[:2]
        s = self.size / max(h, w)
        nh, nw = max(1, round(h * s)), max(1, round(w * s))
        resized = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
        canvas = np.empty((self.size, self.size, 3), np.uint8)
        canvas[:] = (self.mean[::-1] * 255).astype(np.uint8)          # pad with the mean colour
        canvas[:nh, :nw] = resized
        return canvas, nh, nw

    def __call__(self, bgr):
        """BGR frame -> (class map at model resolution, resized BGR view, model-only seconds)."""
        sq, nh, nw = self.letterbox(bgr)
        rgb = cv2.cvtColor(sq, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = ((rgb - self.mean) / self.std).transpose(2, 0, 1)[None].copy()
        t0 = time.perf_counter()
        low = self.sess.run(None, {"image": x})[0][0]                    # (7, h, w) low-res logits
        t_model = time.perf_counter() - t0
        up = np.stack([cv2.resize(c, (self.size, self.size), interpolation=cv2.INTER_LINEAR) for c in low])
        pred = up.argmax(0)[:nh, :nw].astype(np.uint8)                   # drop the letterbox padding
        return pred, sq[:nh, :nw], t_model

    def report(self, pred):
        total = pred.size
        areas = {self.names[c]: 100.0 * int((pred == c).sum()) / total for c in range(1, len(self.names))}
        return 100.0 * int((pred > 0).sum()) / total, areas


def render(view_bgr, pred, names, caption=""):
    """Blend defects onto the image and add a caption + colour legend underneath."""
    v = view_bgr.astype(np.float32)
    m = (pred > 0)[..., None]
    img = np.where(m, 0.45 * v + 0.55 * PALETTE_BGR[pred].astype(np.float32), v).astype(np.uint8)
    w = img.shape[1]
    bar = np.full((44, w, 3), 255, np.uint8)
    cv2.putText(bar, caption, (6, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    x = 6
    for c in range(1, len(names)):
        cv2.rectangle(bar, (x, 25), (x + 12, 37), tuple(int(v) for v in PALETTE_BGR[c]), -1)
        cv2.putText(bar, names[c], (x + 15, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 0), 1, cv2.LINE_AA)
        x += 20 + 7 * len(names[c])
    return np.vstack([img, bar])


# ============================================================================ camera sources
class PiCamera:
    def __init__(self, w, h):
        from picamera2 import Picamera2                                   # apt install python3-picamera2
        self.cam = Picamera2()
        # "RGB888" in picamera2 gives arrays in B,G,R byte order -> already OpenCV's BGR
        self.cam.configure(self.cam.create_video_configuration(main={"size": (w, h), "format": "RGB888"}))
        self.cam.start()
        time.sleep(1.0)                                                    # let exposure settle
        self.name = "picamera2"

    def read(self):
        return self.cam.capture_array("main")

    def close(self):
        self.cam.stop()


class CvCamera:
    def __init__(self, src, w, h):
        self.cap = cv2.VideoCapture(src)
        if isinstance(src, int):
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
            self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)                      # always process the newest frame
        if not self.cap.isOpened():
            raise RuntimeError(f"cannot open camera/video {src!r}")
        self.name = f"opencv:{src}"

    def read(self):
        ok, frame = self.cap.read()
        return frame if ok else None

    def close(self):
        self.cap.release()


def open_camera(spec, w, h):
    if spec in ("auto", "pi"):
        try:
            return PiCamera(w, h)
        except Exception as e:
            if spec == "pi":
                raise
            print(f"Pi camera not available ({type(e).__name__}: {e}); trying USB webcam 0")
            spec = "usb:0"
    if spec.startswith("usb:"):
        return CvCamera(int(spec[4:]), w, h)
    return CvCamera(spec, w, h)                                            # video file or stream URL


# ============================================================================ live web view
class LiveView:
    """Tiny MJPEG server so a headless Pi can be watched from a laptop/phone browser."""

    def __init__(self, port):
        self.cond, self.jpg, self.n = threading.Condition(), None, 0
        live = self

        class Handler(BaseHTTPRequestHandler):
            def do_GET(self):
                if self.path == "/":
                    body = (b"<html><head><title>Concrete inspection</title></head>"
                            b"<body style='margin:0;background:#111'>"
                            b"<img src='/stream' style='width:100%;max-width:1024px'></body></html>")
                    self.send_response(200)
                    self.send_header("Content-Type", "text/html")
                    self.send_header("Content-Length", str(len(body)))
                    self.end_headers()
                    self.wfile.write(body)
                    return
                if self.path != "/stream":
                    self.send_error(404)
                    return
                self.send_response(200)
                self.send_header("Content-Type", "multipart/x-mixed-replace; boundary=frame")
                self.end_headers()
                seen = -1
                try:
                    while True:
                        with live.cond:
                            live.cond.wait_for(lambda: live.n != seen, timeout=10)
                            jpg, seen = live.jpg, live.n
                        if jpg is None:
                            continue
                        self.wfile.write(b"--frame\r\nContent-Type: image/jpeg\r\nContent-Length: "
                                         + str(len(jpg)).encode() + b"\r\n\r\n" + jpg + b"\r\n")
                except (BrokenPipeError, ConnectionResetError):
                    pass

            def log_message(self, *a):
                pass

        self.server = ThreadingHTTPServer(("0.0.0.0", port), Handler)
        self.server.daemon_threads = True
        threading.Thread(target=self.server.serve_forever, daemon=True).start()
        print(f"live view: http://<this-pi's-address>:{port}  (find the address with: hostname -I)")

    def push(self, img):
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
        if ok:
            with self.cond:
                self.jpg, self.n = buf.tobytes(), self.n + 1
                self.cond.notify_all()


# ============================================================================ main loops
def log_row(writer, fh, name, t_model, t_total, defect, areas):
    writer.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), name, f"{1000 * t_model:.0f}", f"{1000 * t_total:.0f}",
                     f"{defect:.2f}"] + [f"{v:.2f}" for v in areas.values()])
    fh.flush()


def print_report(name, t_model, t_total, defect, areas):
    print(f"\n{name}: model {1000 * t_model:.0f} ms, total {1000 * t_total:.0f} ms | defect area {defect:.2f}%")
    for k, v in areas.items():
        if v > 0:
            print(f"   {k:<14s} {v:6.2f}%")


def run_images(seg, inputs, out, writer, fh):
    files = []
    for p in map(Path, inputs):
        files += sorted(f for f in p.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"}) \
            if p.is_dir() else [p]
    for f in files:
        bgr = cv2.imread(str(f))
        if bgr is None:
            print(f"skip {f} (unreadable)")
            continue
        t0 = time.perf_counter()
        pred, view, t_model = seg(bgr)
        t_total = time.perf_counter() - t0
        defect, areas = seg.report(pred)
        cv2.imwrite(str(out / f"{f.stem}_overlay.jpg"),
                    render(view, pred, seg.names, f"{f.name} | defects {defect:.1f}%"))
        print_report(f.name, t_model, t_total, defect, areas)
        log_row(writer, fh, f.name, t_model, t_total, defect, areas)


def run_camera(seg, args, out, writer, fh):
    w, h = map(int, args.capture.lower().split("x"))
    cam = open_camera(args.camera, w, h)
    live = LiveView(args.serve) if args.serve else None
    print(f"camera: {cam.name} | Ctrl+C to stop")
    n, last_save, fps = 0, -1e9, 0.0
    try:
        while not args.max_frames or n < args.max_frames:
            frame = cam.read()
            if frame is None:
                print("no more frames")
                break
            t0 = time.perf_counter()
            pred, view, t_model = seg(frame)
            t_total = time.perf_counter() - t0
            n += 1
            fps = 0.8 * fps + 0.2 / t_total if n > 1 else 1 / t_total
            defect, areas = seg.report(pred)
            top = max(areas, key=areas.get)
            caption = (f"{time.strftime('%H:%M:%S')}  {fps:.2f} fps  defects {defect:.1f}%"
                       + (f"  (most: {top} {areas[top]:.1f}%)" if defect > 0 else ""))
            img = render(view, pred, seg.names, caption)
            if live:
                live.push(img)
            if args.show:
                cv2.imshow("concrete inspection", img)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            now = time.time()
            alert = args.alert > 0 and defect >= args.alert
            if now - last_save >= args.save_every or alert:
                last_save = now
                stem = f"frame_{time.strftime('%Y%m%d_%H%M%S')}_{n:05d}"
                cv2.imwrite(str(out / f"{stem}_overlay.jpg"), img)
                cv2.imwrite(str(out / f"{stem}.jpg"), frame)
                log_row(writer, fh, stem, t_model, t_total, defect, areas)
                print(f"{stem}: {1000 * t_total:.0f} ms | defect area {defect:.2f}%"
                      + ("  << ALERT" if alert else ""))
    except KeyboardInterrupt:
        print("\nstopped")
    finally:
        cam.close()
        if args.show:
            cv2.destroyAllWindows()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("inputs", nargs="*", help="image files or folders")
    ap.add_argument("--model", default=str(Path(__file__).with_name("student_int8.onnx")))
    ap.add_argument("--meta", default=None, help="default: student_meta.json next to the model")
    ap.add_argument("--camera", nargs="?", const="auto", default=None,
                    help="auto (default) | pi | usb:N | path/to/video.mp4")
    ap.add_argument("--capture", default="1024x768", help="camera capture size WxH")
    ap.add_argument("--serve", type=int, default=0, help="port for the live browser view (0 = off)")
    ap.add_argument("--show", action="store_true", help="show a window (needs a desktop)")
    ap.add_argument("--save_every", type=float, default=10.0, help="seconds between saved frames (0 = all)")
    ap.add_argument("--alert", type=float, default=0.0, help="also save any frame with defect area >= this %%")
    ap.add_argument("--max_frames", type=int, default=0)
    ap.add_argument("--out", default="results")
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args()
    if not args.inputs and not args.camera:
        ap.error("give image files/folders, or --camera")

    seg = Segmenter(args.model, args.meta, args.threads)
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    log_path = out / "log.csv"
    new = not log_path.exists()
    with open(log_path, "a", newline="") as fh:
        writer = csv.writer(fh)
        if new:
            writer.writerow(["time", "source", "model_ms", "total_ms", "defect_%"]
                            + [f"{n}_%" for n in seg.names[1:]])
        if args.inputs:
            run_images(seg, args.inputs, out, writer, fh)
        if args.camera:
            try:
                run_camera(seg, args, out, writer, fh)
            except RuntimeError as e:
                raise SystemExit(f"ERROR: {e}. Check the camera cable/connection "
                                 "(Pi camera: `rpicam-hello`; USB: `ls /dev/video*`).")


if __name__ == "__main__":
    main()

### Visual comparison: ground truth / teacher / student (int8 ONNX, exactly what the Pi runs)

In [ ]:
import cv2, numpy as np, torch, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from distill import import_repo, build_teacher, teacher_outputs
from pi_infer import Segmenter

CLASS_NAMES = ["Background", "Crack", "Spalling", "Corrosion", "Efflorescence", "Vegetation", "Control Point"]
PALETTE = np.array([[35, 35, 35], [255, 50, 50], [255, 165, 0], [160, 32, 240],
                    [255, 255, 0], [0, 200, 0], [0, 191, 255]], np.uint8)
LEGEND = [mpatches.Patch(color=PALETTE[c] / 255, label=CLASS_NAMES[c]) for c in range(1, 7)]

_, _, _, DINOv3MultiScaleSeg = import_repo(REPO, DATA_ROOT)
teacher = build_teacher(DINOv3MultiScaleSeg, TEACHER_CKPT, 1024, torch.device("cuda"))
amp = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else None
student = Segmenter(f"{PI_DIR}/student_int8.onnx")

pairs = D.list_split("val")
for img_path, lab_path in [pairs[i] for i in (0, 20, 40, 60)]:
    rgb = D._read_rgb(img_path)
    gt = D.lab2class(D._read_rgb(lab_path))
    x = torch.from_numpy(((rgb / 255.0 - D.IMAGENET_MEAN) / D.IMAGENET_STD).transpose(2, 0, 1)).float()[None].cuda()
    t_pred = teacher_outputs(teacher, x, gt.shape, amp)[0].argmax(1)[0].cpu().numpy()
    s_pred, _, _ = student(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))                  # 512 x 512
    s_pred = cv2.resize(s_pred, gt.shape[::-1], interpolation=cv2.INTER_NEAREST)
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    for a, im, title in zip(ax, (rgb, PALETTE[gt], PALETTE[t_pred], PALETTE[s_pred]),
                            ("image", "ground truth", "teacher (DINOv3-L)", "student (MobileNet, int8)")):
        a.imshow(im); a.set_title(title); a.axis("off")
    fig.legend(handles=LEGEND, loc="lower center", ncol=6); plt.show()
del teacher; torch.cuda.empty_cache()

### Camera dry-run
A short video made from validation images stands in for the camera (4:3 frames, like the Pi camera). This runs the same
code path the Pi uses with `--camera`.

In [ ]:
from IPython.display import Image, display
vw = cv2.VideoWriter("/content/test_clip.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 1, (1024, 768))
for img_path, _ in pairs[:10]:
    vw.write(cv2.imread(img_path)[128:896])                  # centre 1024x768 crop = a 4:3 "camera" frame
vw.release()
!rm -rf /content/cam_test
!python pi_infer.py --model {PI_DIR}/student_int8.onnx --camera /content/test_clip.mp4 --save_every 0 --out /content/cam_test
import glob
display(Image(sorted(glob.glob("/content/cam_test/*_overlay.jpg"))[3]))
print(open("/content/cam_test/log.csv").read()[:800])

## 6. Package for the Pi
The zip holds the models, `pi_infer.py` and `setup_pi.sh` (one-time install on the Pi). A copy goes to Drive.

In [ ]:
%%writefile /content/pi_package/setup_pi.sh
#!/bin/bash
# One-time setup on the Raspberry Pi (64-bit Raspberry Pi OS). Run:  bash setup_pi.sh
set -e
[ "$(uname -m)" = "aarch64" ] || { echo "Needs the 64-bit Raspberry Pi OS (uname -m must print aarch64)"; exit 1; }
sudo apt update
# numpy / OpenCV / picamera2 from apt (built for the Pi and for each other)
sudo apt install -y --no-install-recommends python3-venv python3-numpy python3-opencv python3-picamera2
# venv that can see those apt packages; only onnxruntime comes from pip
python3 -m venv --system-site-packages ~/pienv
~/pienv/bin/pip install --upgrade pip
~/pienv/bin/pip install onnxruntime
~/pienv/bin/python -c "import onnxruntime, cv2, numpy; print('OK: onnxruntime', onnxruntime.__version__, '| opencv', cv2.__version__, '| numpy', numpy.__version__)"
echo "Done. Activate with:  source ~/pienv/bin/activate"

In [ ]:
!cp pi_infer.py {PI_DIR}/ && chmod +x {PI_DIR}/setup_pi.sh && ls -la {PI_DIR}
!cd /content && rm -f pi_package.zip && zip -qr pi_package.zip pi_package
!cp /content/pi_package.zip "{DRIVE_DIR}/pi_package.zip"
from google.colab import files
files.download("/content/pi_package.zip")

## On the Raspberry Pi 3B+

1. **Flash** *Raspberry Pi OS Lite (64-bit)* with Raspberry Pi Imager; in its settings enable SSH and enter your Wi-Fi.
2. **Camera:** connect the camera module ribbon cable (or plug in a USB webcam). Check it with `rpicam-hello --list-cameras`
   (Pi camera) or `ls /dev/video*` (USB).
3. **Copy and install** (from your laptop, then on the Pi):
   ```bash
   scp pi_package.zip pi@<pi-address>:~
   ssh pi@<pi-address>
   unzip pi_package.zip && cd pi_package
   bash setup_pi.sh                     # one time, a few minutes
   source ~/pienv/bin/activate
   ```
4. **Run**
   ```bash
   python pi_infer.py some_photo.jpg                     # a photo  -> results/some_photo_overlay.jpg
   python pi_infer.py --camera --serve 8080              # live camera; open http://<pi-address>:8080 in a browser
   python pi_infer.py --camera --save_every 5 --alert 2  # save a frame every 5 s, and any frame with >= 2 % defect area
   ```
   Stop with Ctrl+C. Results are in `results/` (overlays, raw frames, `log.csv`).

**Tips**
* Speed: read the `fps` in the overlay caption. If int8 is not faster than fp32 on your Pi, use `--model student_fp32.onnx`.
* A heatsink/fan helps: the Pi 3B+ slows down when it gets hot under constant load.
* S2DS photos are close-ups of concrete surfaces. Hold the camera at a similar distance (roughly 0.5–2 m, filling the frame
  with the surface) for results like the validation set.